# Apache Beam Data Engineering Exercise

**Student**: [Your Name]
**Course**: Data Engineering
**Assignment**: Apache Beam Features Demonstration

---

## Features Demonstrated

1. ✅ **Composite Transforms** - Reusable transform combinations
2. ✅ **Pipeline I/O** - Reading and writing data
3. ✅ **ParDo** - Parallel processing with DoFn
4. ✅ **Windowing** - Time-based data grouping
5. ✅ **Map** - Element-wise transformations
6. ✅ **Filter** - Conditional selection
7. ✅ **Partition** - Data splitting

**Scenario**: E-commerce Transaction Processing System

## Step 1: Install Apache Beam

In [ ]:
!pip install apache-beam -q

## Step 2: Import Libraries

In [ ]:
import apache_beam as beam
from apache_beam import window
from apache_beam.options.pipeline_options import PipelineOptions
import json
from datetime import datetime, timedelta
import random

print(f"✓ Apache Beam version: {beam.__version__}")

## Step 3: Data Generation

Generate 200 sample e-commerce transactions with realistic data.

In [ ]:
def generate_sample_transactions(num_transactions=200):
    """Generate sample e-commerce transactions"""
    products = [
        ('Laptop', 1200, 'Electronics'),
        ('Smartphone', 800, 'Electronics'),
        ('Headphones', 150, 'Electronics'),
        ('Book', 25, 'Books'),
        ('Desk Chair', 300, 'Furniture'),
        ('Coffee Maker', 80, 'Appliances'),
        ('Running Shoes', 120, 'Sports'),
        ('Backpack', 60, 'Accessories'),
        ('Monitor', 400, 'Electronics'),
        ('Keyboard', 100, 'Electronics')
    ]
    
    customers = [f'CUST{str(i).zfill(4)}' for i in range(1, 51)]
    transactions = []
    base_time = datetime.now()
    
    for i in range(num_transactions):
        product_name, price, category = random.choice(products)
        quantity = random.randint(1, 5)
        timestamp = base_time - timedelta(hours=random.randint(0, 24), minutes=random.randint(0, 59))
        
        transaction = {
            'transaction_id': f'TXN{str(i+1).zfill(6)}',
            'customer_id': random.choice(customers),
            'product_name': product_name,
            'category': category,
            'price': price,
            'quantity': quantity,
            'total_amount': price * quantity,
            'timestamp': timestamp.isoformat(),
            'payment_method': random.choice(['Credit Card', 'Debit Card', 'PayPal', 'Cash']),
            'region': random.choice(['North', 'South', 'East', 'West'])
        }
        transactions.append(transaction)
    
    return transactions

# Generate and save data
transactions = generate_sample_transactions(200)
with open('transactions.json', 'w') as f:
    for txn in transactions:
        f.write(json.dumps(txn) + '\n')

print(f"✓ Generated {len(transactions)} transactions")
print("\nSample transaction:")
print(json.dumps(transactions[0], indent=2))

## Step 4: ParDo - Parallel Processing

**Feature #3**: ParDo with custom DoFn classes for parallel data processing.

In [ ]:
class ParseTransactionFn(beam.DoFn):
    """Parse JSON transaction strings"""
    def process(self, element):
        try:
            yield json.loads(element)
        except json.JSONDecodeError as e:
            print(f"Error: {e}")

class EnrichTransactionFn(beam.DoFn):
    """Enrich transactions with computed fields"""
    def process(self, element):
        total = element['total_amount']
        
        # Calculate discount
        if total > 1000:
            discount_rate = 0.15
        elif total > 500:
            discount_rate = 0.10
        elif total > 200:
            discount_rate = 0.05
        else:
            discount_rate = 0.0
        
        element['discount_rate'] = discount_rate
        element['discount_amount'] = total * discount_rate
        element['final_amount'] = total - element['discount_amount']
        
        # Customer tier
        if total > 1000:
            element['customer_tier'] = 'Premium'
        elif total > 500:
            element['customer_tier'] = 'Gold'
        elif total > 200:
            element['customer_tier'] = 'Silver'
        else:
            element['customer_tier'] = 'Bronze'
        
        yield element

class ExtractCategoryAmountFn(beam.DoFn):
    """Extract category and amount as key-value pairs"""
    def process(self, element):
        yield (element['category'], element['final_amount'])

class AddTimestampFn(beam.DoFn):
    """Add timestamps for windowing"""
    def process(self, element):
        timestamp = datetime.fromisoformat(element['timestamp'])
        unix_timestamp = timestamp.timestamp()
        yield window.TimestampedValue(element, unix_timestamp)

print("✓ ParDo DoFn classes defined")

## Step 5: Composite Transforms

**Feature #1**: Composite transforms that combine multiple operations into reusable components.

In [ ]:
class AnalyzeCustomerSpending(beam.PTransform):
    """Composite Transform: Analyze customer spending"""
    def expand(self, pcoll):
        return (
            pcoll
            | 'Extract Customer Amount' >> beam.Map(lambda x: (x['customer_id'], x['final_amount']))
            | 'Group By Customer' >> beam.GroupByKey()
            | 'Calculate Stats' >> beam.Map(
                lambda x: {
                    'customer_id': x[0],
                    'total_spent': sum(x[1]),
                    'num_transactions': len(list(x[1])),
                    'avg_transaction': sum(x[1]) / len(list(x[1]))
                }
            )
        )

class CategorySalesAnalysis(beam.PTransform):
    """Composite Transform: Analyze category sales"""
    def expand(self, pcoll):
        return (
            pcoll
            | 'Extract Category Amount' >> beam.ParDo(ExtractCategoryAmountFn())
            | 'Group By Category' >> beam.GroupByKey()
            | 'Sum By Category' >> beam.Map(
                lambda x: {
                    'category': x[0],
                    'total_sales': sum(x[1]),
                    'num_items': len(list(x[1])),
                    'avg_sale': sum(x[1]) / len(list(x[1]))
                }
            )
        )

print("✓ Composite transforms defined")

## Step 6: Partition Function

**Feature #7**: Partition to split data into multiple outputs.

In [ ]:
def partition_by_amount(element, num_partitions):
    """Partition: 0=Small(<$200), 1=Medium($200-$1000), 2=Large(>$1000)"""
    amount = element['final_amount']
    if amount < 200:
        return 0
    elif amount <= 1000:
        return 1
    else:
        return 2

print("✓ Partition function defined")

## Step 7: Main Pipeline

Demonstrates:
- **Feature #2**: Pipeline I/O (Read/Write)
- **Feature #5**: Map transformations
- **Feature #6**: Filter operations
- All features combined

In [ ]:
print("Running main pipeline...\n")

options = PipelineOptions()
with beam.Pipeline(options=options) as pipeline:
    
    # PIPELINE I/O: Read
    raw_transactions = pipeline | 'Read' >> beam.io.ReadFromText('transactions.json')
    
    # PARDO: Parse and enrich
    parsed = raw_transactions | 'Parse' >> beam.ParDo(ParseTransactionFn())
    enriched = parsed | 'Enrich' >> beam.ParDo(EnrichTransactionFn())
    
    # MAP: Create summaries
    summaries = enriched | 'Map Summaries' >> beam.Map(
        lambda x: f"{x['transaction_id']}: {x['customer_id']} spent ${x['final_amount']:.2f}"
    )
    
    # FILTER: High-value transactions
    high_value = enriched | 'Filter High Value' >> beam.Filter(lambda x: x['final_amount'] > 500)
    
    # PARTITION: Split by size
    small, medium, large = enriched | 'Partition' >> beam.Partition(partition_by_amount, 3)
    
    # COMPOSITE TRANSFORMS
    customer_analysis = enriched | 'Customer Analysis' >> AnalyzeCustomerSpending()
    category_analysis = enriched | 'Category Analysis' >> CategorySalesAnalysis()
    
    # PIPELINE I/O: Write outputs
    enriched | 'Write Enriched' >> beam.io.WriteToText('output/enriched', file_name_suffix='.json', shard_name_template='')
    summaries | 'Write Summaries' >> beam.io.WriteToText('output/summaries', file_name_suffix='.txt', shard_name_template='')
    high_value | 'Write High Value' >> beam.io.WriteToText('output/high_value', file_name_suffix='.json', shard_name_template='')
    small | 'Write Small' >> beam.io.WriteToText('output/small', file_name_suffix='.json', shard_name_template='')
    medium | 'Write Medium' >> beam.io.WriteToText('output/medium', file_name_suffix='.json', shard_name_template='')
    large | 'Write Large' >> beam.io.WriteToText('output/large', file_name_suffix='.json', shard_name_template='')
    customer_analysis | 'Write Customer' >> beam.io.WriteToText('output/customer_analysis', file_name_suffix='.json', shard_name_template='')
    category_analysis | 'Write Category' >> beam.io.WriteToText('output/category_analysis', file_name_suffix='.json', shard_name_template='')

print("✓ Main pipeline completed!")

## Step 8: Windowing Pipeline

**Feature #4**: Windowing with Fixed, Sliding, and Session windows.

In [ ]:
print("Running windowing pipeline...\n")

options = PipelineOptions()
with beam.Pipeline(options=options) as pipeline:
    
    transactions = (
        pipeline
        | 'Read2' >> beam.io.ReadFromText('transactions.json')
        | 'Parse2' >> beam.ParDo(ParseTransactionFn())
        | 'Enrich2' >> beam.ParDo(EnrichTransactionFn())
    )
    
    timestamped = transactions | 'Add Timestamps' >> beam.ParDo(AddTimestampFn())
    
    # WINDOWING: Fixed Windows (1 hour)
    hourly_sales = (
        timestamped
        | 'Fixed Windows' >> beam.WindowInto(window.FixedWindows(60 * 60))
        | 'Extract Amount' >> beam.Map(lambda x: x['final_amount'])
        | 'Sum Hourly' >> beam.CombineGlobally(sum).without_defaults()
        | 'Format Hourly' >> beam.Map(lambda x: f"Hourly sales: ${x:.2f}")
    )
    
    # WINDOWING: Sliding Windows (2hr window, 1hr slide)
    sliding_sales = (
        timestamped
        | 'Sliding Windows' >> beam.WindowInto(window.SlidingWindows(60 * 60 * 2, 60 * 60))
        | 'Extract Category' >> beam.Map(lambda x: (x['category'], x['final_amount']))
        | 'Group Category' >> beam.GroupByKey()
        | 'Sum Category' >> beam.Map(lambda x: {'category': x[0], 'total_sales': sum(x[1])})
    )
    
    # WINDOWING: Session Windows (30min gap)
    sessions = (
        timestamped
        | 'Session Windows' >> beam.WindowInto(window.Sessions(30 * 60))
        | 'Extract Session' >> beam.Map(lambda x: (x['customer_id'], 1))
        | 'Count Sessions' >> beam.GroupByKey()
        | 'Format Sessions' >> beam.Map(lambda x: {'customer_id': x[0], 'transactions': len(list(x[1]))})
    )
    
    # Write windowed results
    hourly_sales | 'Write Hourly' >> beam.io.WriteToText('output/hourly_sales', file_name_suffix='.txt', shard_name_template='')
    sliding_sales | 'Write Sliding' >> beam.io.WriteToText('output/sliding_sales', file_name_suffix='.json', shard_name_template='')
    sessions | 'Write Sessions' >> beam.io.WriteToText('output/sessions', file_name_suffix='.json', shard_name_template='')

print("✓ Windowing pipeline completed!")

## Step 9: View Results

Display the outputs from our pipelines.

In [ ]:
import os

def show_file(filename, lines=5):
    if os.path.exists(filename):
        print(f"\n{'='*60}")
        print(f"{filename}:")
        print('='*60)
        with open(filename, 'r') as f:
            for i, line in enumerate(f):
                if i >= lines:
                    print("...")
                    break
                print(line.strip())
    else:
        print(f"File not found: {filename}")

# Display outputs
print("\n" + "="*60)
print("PIPELINE OUTPUTS")
print("="*60)

show_file('output/customer_analysis.json')
show_file('output/category_analysis.json')
show_file('output/summaries.txt')
show_file('output/hourly_sales.txt')

# List all output files
print(f"\n{'='*60}")
print("All output files created:")
print('='*60)
if os.path.exists('output'):
    for file in sorted(os.listdir('output')):
        print(f"  ✓ {file}")
else:
    print("Output directory not found")

## Summary

### ✅ All Features Demonstrated

1. ✅ **Composite Transforms** - `AnalyzeCustomerSpending`, `CategorySalesAnalysis`
2. ✅ **Pipeline I/O** - Read from `transactions.json`, Write to 11 output files
3. ✅ **ParDo** - 4 DoFn classes: Parse, Enrich, Extract, AddTimestamp
4. ✅ **Windowing** - Fixed (1hr), Sliding (2hr/1hr), Session (30min)
5. ✅ **Map** - Transaction summaries and field extraction
6. ✅ **Filter** - High-value transactions (>$500)
7. ✅ **Partition** - 3-way split: Small/Medium/Large

### 📊 Outputs Generated

- `enriched.json` - All transactions with computed fields
- `summaries.txt` - Human-readable transaction summaries
- `high_value.json` - Filtered high-value transactions
- `small/medium/large.json` - Partitioned transactions
- `customer_analysis.json` - Customer spending statistics
- `category_analysis.json` - Category sales statistics
- `hourly_sales.txt` - Fixed window aggregations
- `sliding_sales.json` - Sliding window aggregations
- `sessions.json` - Session window groupings

### 🎯 Key Takeaways

- **ParDo** enables parallel processing of each element
- **Composite Transforms** create reusable pipeline components
- **Windowing** groups data by time for temporal analysis
- **Map/Filter/Partition** provide flexible data transformations
- **Pipeline I/O** handles data ingestion and export

---

**Assignment Complete!** ✨